### 🔰PyTorchでニューラルネットワーク基礎 #36【可変長BERTタイプ事前学習】
* Qiitaの記事と連動しています
* 系列長を可変にしたMLMによるBERT風モデルの事前学習
* 各種ファイルの保存先は環境によって適宜変更してください

**扱う内容**
1. BERTタイプの事前学習
2. Datasetクラスのカスタマイズ
3. DataLoaderクラスのカスタマイズ
    * 可変長に対応する形でMLMを構成する
    * collate関数もHuggingFaceライブラリを用いない
4. step数による学習の管理

**トークナイザー**
* ファイル名：greek_unigram_tokenizer_3k.json
* wikipediaのギリシア神話分野から収集したテキストを利用してunigram lm で学習したもの。
* 語彙数3000トークン

**データについて**
入力されるデータが可変長、つまり、idsの長さが異なる状況での事前学習を想定しているので、わざと1文1行となっています。
* ファイル名：greek_data_unigram_3k.jsonl
* wikipediaのギリシア神話分野から収集したテキストを１文毎に分割したテキストデータ。
* idsは、文頭に\<bos\>、文末に\<eos\>トークンを挿入しています。


In [1]:
import torch
import torch.nn as nn
import random
from tokenizers import Tokenizer

# カスタマイズする部分
from torch.utils.data import Dataset, DataLoader  # カスタムクラス作成
from torch.nn.utils.rnn import pad_sequence       # paddingで利用
from functools import partial                     # collate_fn関数で利用（部分適用）


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
data_filename = "./data/greek_data_unigram_3k.jsonl"                # 学習データ
tokenizer_filename = "./tokenizer/greek_unigram_tokenizer_3k.json"  # トークナイザー
pretrain_filename = "./model/greek_unigram_3k_step.model"           # 保存する事前学習モデル名

# トークナイザーの確認
tokenizer = Tokenizer.from_file(tokenizer_filename)

print("特殊トークンID:")
print(f"<pad>: {tokenizer.token_to_id('<pad>')}")
print(f"<bos>: {tokenizer.token_to_id('<bos>')}")
print(f"<eos>: {tokenizer.token_to_id('<eos>')}")
print(f"<unk>: {tokenizer.token_to_id('<unk>')}")
print(f"<mask>: {tokenizer.token_to_id('<mask>')}")
print(f"size: {tokenizer.get_vocab_size()}")

特殊トークンID:
<pad>: 0
<bos>: 1
<eos>: 2
<unk>: 3
<mask>: 4
size: 3000


In [3]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = tokenizer.get_vocab_size()
        self.seq_len = 64
        self.d_model = 64
        self.nhead = 4
        self.dim_feedforward = 256
        self.num_layers = 6
        self.dropout = 0.1
        
        # 特殊トークンID
        self.pad_token_id = tokenizer.token_to_id("<pad>")
        self.mask_token_id = tokenizer.token_to_id("<mask>")
        self.bos_token_id = tokenizer.token_to_id("<bos>")
        self.eos_token_id = tokenizer.token_to_id("<eos>")
        self.unk_token_id = tokenizer.token_to_id("<unk>")

        # 特殊トークンのセット
        self.special_tokens_set = {
            self.pad_token_id,
            self.mask_token_id,
            self.bos_token_id,
            self.eos_token_id,
            self.unk_token_id,
        }
        
        # 通常トークンのリスト special_tokenを除くトークンのリスト（MLMランダム置換用）
        self.normal_tokens_list = [
            i for i in range(self.vocab_size) 
            if i not in self.special_tokens_set
        ]
        
        # 学習設定 (使うと便利かも、今回は一部だけ使ってみました)
        self.batch_size = 64
        self.learning_rate = 0.001 # 学習率で計算
        self.num_epochs = 100
        self.mask_prob = 0.15
        self.max_grad_norm = 1.0
        # PyTorchの仕様 ID= -100　は損失計算時に除外されるマスクID
        self.ignore_index = -100

# MLM用の出力部分
class MLMHead(nn.Module):
    def __init__(self, config: ModelConfig, embedding_weight):
        super().__init__()
        self.fc = nn.Linear(config.d_model, config.d_model)
        self.act = nn.GELU()
        self.ln = nn.LayerNorm(config.d_model)

        self.classification_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.classification_head.weight = embedding_weight  # weight tying

        self.bias = nn.Parameter(torch.zeros(config.vocab_size))

    def forward(self, x):
        x = self.fc(x)
        x = self.act(x)
        x = self.ln(x)
        x = self.classification_head(x) + self.bias
        return x

# MLM用のモデル（出力層を変更）
class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        
        # 埋め込み層
        self.token_embedding = nn.Embedding(
            num_embeddings=config.vocab_size, 
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id
        )
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers, enable_nested_tensor=False)
        
        # MLM用の出力層：BERT風レイヤー　各トークン位置で語彙全体を予測
        self.mlm_head = MLMHead(config, self.token_embedding.weight)
        #self.mlm_head = nn.Linear(config.d_model, config.vocab_size)
    
    def forward(self, x):
        # マスクの作成
        src_key_padding_mask = (x == self.config.pad_token_id)
        
        # 埋め込み
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(torch.arange(x.size(1), device=x.device))
        x = tok_emb + pos_emb.unsqueeze(0)
        
        x = self.layer_norm(x)
        x = self.dropout(x)
        
        # Transformer Encoder
        h = self.transformer_encoder(x, src_key_padding_mask=src_key_padding_mask)
        
        # MLM予測：各位置で語彙全体での確率を出力
        logits = self.mlm_head(h)  # [batch, seq_len, vocab_size]
        
        return logits

config = ModelConfig(tokenizer)
model = DNN(config).to(device)

### データの読み込み
* tokenizerのIDでトークナイズしたデータを読み込む。
* tokenizerが変わるとIDも変わるので注意

In [4]:
# データファイルの読み込みJSONファイルを読み込む
import pandas as pd

data = pd.read_json(data_filename, lines=True)

print(f"データの構造:{data.keys()}")
print(f"データ数: {len(data['ids'])}")
print(f"サンプル長さ: {[len(d) for d in data['ids'][:5]]}")  # 可変長を確認

データの構造:Index(['text', 'ids'], dtype='object')
データ数: 2979
サンプル長さ: [23, 17, 18, 10, 47]


In [5]:
def create_mlm_sample(
    original_ids: list,
    config: ModelConfig,
    mask_prob: float = 0.15,
    ):
    input_ids = original_ids.copy()
    labels = [config.ignore_index] * len(original_ids)   # config.ignore_index= -100

    special_tokens_set = config.special_tokens_set # {0,1,2,3,4}  # <pad>, <bos>, <eos>, <unk>, <mask>になる予定
    normal_tokens_list = config.normal_tokens_list  # range(5,10_000) # 語彙IDのspecial_tokensを除いたもの


    # 句読点も除外 (マスクに句読点を入れる結果になりやすかったので今回あえて削除しました)
    no_mask_tokens = ["、", "。", ",", ".", "，", "．"]
    no_mask_ids = set(
        token_id for token_id in [tokenizer.token_to_id(tok) for tok in no_mask_tokens]
        if token_id is not None
    )
    special_tokens_set.update(no_mask_ids)

    for i in range(len(original_ids)):
        # original_ids[i]: int型
        token_id = original_ids[i]
        
        if token_id in special_tokens_set:
            continue
        
        # 15%未満で他のトークンへ置き換え
        # 80%は<mask> mask_token_id
        # 10%は特殊トークを除く別トークン random.choice()
        # 10%はそのまま pass
        if random.random() < mask_prob:
            labels[i] = original_ids[i]
            
            rand = random.random()
            if rand < 0.8:
                input_ids[i] = config.mask_token_id
            elif rand < 0.9:
                input_ids[i] = random.choice(normal_tokens_list) 
            else:
                pass
    
    return input_ids, labels




def collate_fn_mlm(batch_data, config):
    """
    バッチ内で動的にパディングを行う
    
    Args:
        batch_data: [{'ids': [...]}, {'ids': [...]}, ...]
        tokenizer: トークナイザー
   
    Returns:
        input_ids: [batch_size, max_len_in_batch]
        labels: [batch_size, max_len_in_batch]
    """
    mask_token_id = config.mask_token_id
    pad_token_id = config.pad_token_id
    #bos_token_id = config.bos_token_id
    #eos_token_id = config.eos_token_id


    # MLMマスクを適用
    input_ids_list = []
    labels_list = []
    
    for item in batch_data:
        original_ids = item
        input_ids, labels = create_mlm_sample(original_ids, config=config, mask_prob=config.mask_prob)
        input_ids_list.append(input_ids)
        labels_list.append(labels)
    
    # バッチ内の最大長を取得, config.seq_len=64がmax
    max_len = min(max(len(ids) for ids in input_ids_list), config.seq_len)

    # パディング
    padded_input_ids = []
    padded_labels = []
    
    for input_ids, labels in zip(input_ids_list, labels_list):
        input_ids = input_ids[:max_len]   # max_lenで切り詰め
        labels    = labels[:max_len]      # max_lenで切り詰め
        padding_length = max_len - len(input_ids)
        
        # input_idsをパディング
        padded_input = input_ids + [pad_token_id] * padding_length
        padded_input_ids.append(padded_input)
        
        # labelsをパディング（ignore_index=-100でパディング）
        padded_label = labels + [config.ignore_index] * padding_length
        padded_labels.append(padded_label)
    
    # Tensorに変換
    input_ids_tensor = torch.LongTensor(padded_input_ids)
    labels_tensor = torch.LongTensor(padded_labels)
    
    return {
        "input_ids": input_ids_tensor,
        "label_ids": labels_tensor
    }

# DataLoaderで利用するためのラップ処理
collate_wrapper = partial(collate_fn_mlm, config=config)

In [6]:
class MLMDataset(Dataset):
    def __init__(self, data):
        # 入力するデータによって適宜修正
        self.ids = data["ids"].tolist()
   
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, idx):
        return self.ids[idx]

In [8]:
dataloader = DataLoader(
    dataset,
    batch_size=config.batch_size,
    shuffle=True,
    drop_last=True,
    collate_fn=collate_wrapper, # 動的パディング
    num_workers=0,
    #persistent_workers=True,    # epochごとにworkerを再生成せず使い回す（起動コスト削減と高速化）効果はやや不明
)

In [10]:
# step数で管理するためのデータローダー関数
def infinite_loader(dataloader):
    while True:
        for batch in dataloader:
            yield batch

In [11]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=config.ignore_index)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [12]:
def accuracy(y, t, ignore_index=config.ignore_index):
    """
    マスク位置での予測精度を計算
    ※この関数内では勾配計算を行わない
    """
    with torch.no_grad():
        preds = torch.argmax(y, dim=-1)
        mask = (t != ignore_index)
        correct = (preds == t) & mask

        num_correct = correct.sum().item()
        num_total = mask.sum().item()
        acc = (num_correct / num_total) if num_total > 0 else 0.0

        return acc

In [ ]:
from tqdm import tqdm

data_iter = infinite_loader(dataloader)  # epochではなく、step数で計測
max_iters = 70_000 # step数 (更新回数) = epoch数 (LOOPの回数)× ミニバッチ分割数

pbar = tqdm(range(max_iters))
model.train()                            # trainモードを明示
for step in pbar:
    batch = next(data_iter)
    x = batch["input_ids"].to(device)
    t = batch["label_ids"].to(device)
    optimizer.zero_grad()
    y = model(x)
    loss = criterion(y.view(-1, config.vocab_size), t.view(-1))
    acc = accuracy(y, t)    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.max_grad_norm)  # 勾配クリップ
    optimizer.step()

    # set_postfixは辞書が引数になる
    pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc":f"{acc:.3f}"})
    if (step+1)%1000 == 0:
        tqdm.write(f"{step+1}-step:\tloss:{loss.item():.3f}\tacc:{acc:.3f}")

In [ ]:
# 保存
torch.save({
    "model_state_dict": model.state_dict(),
    "config": config.__dict__,  # configも一緒に保存
}, pretrain_filename)

# 復元
checkpoint = torch.load(pretrain_filename)
config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])
model = DNN(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

### マスク埋め判定
* マスク埋めの関数は第31回のBERTタイプの事前学習で利用したものをそのまま使います。

In [16]:
def encode_text_with_special_tokens(text, tokenizer, config):
    """
    テキストをtokenizerに従い、idsへ変換する関数
    text: 例 織田信<mask>は桶狭間の戦いで... : <mask>は1個に限定しておく
    tokenizer: 学習時に使った tokenizer
    戻り値: (1, seq_len) = (1, 64) の torch.LongTensor input_ids
    """
    ids = tokenizer.encode(text).ids                           # 文字列をID化
    ids = [config.bos_token_id] + ids + [config.eos_token_id]  # <bos>, <eos> を追加
    ids = ids[:config.seq_len]                                 # 長すぎる場合は切る
    if len(ids) < config.seq_len:                              # 足りない場合は<pad>
        ids = ids + [config.pad_token_id] * (config.seq_len - len(ids))

    # LongTensorに変換して出力
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    return input_ids

@torch.inference_mode()
def predict_mask_topk(model, tokenizer, config, text, topk=5):
    """
    text中の <mask> 位置を見つけて，上位候補を返す
    """
    model.eval()
    input_ids = encode_text_with_special_tokens(text, tokenizer, config)  # textをエンコード化<bos>や<eos>もついているぞ
    logits = model(input_ids).to(device)           # モデル出力: (1, seq_len, vocab_size) = (1, 64, 2000)
    mask_positions = (input_ids[0] == config.mask_token_id).nonzero(as_tuple=True)[0]  # <mask> の位置を探す
    pos = mask_positions.item()                    # mask_positionはtorch.Tensorなので数値に戻す
    mask_logits = logits[0, pos]                   # maski_position位置の語彙方向のスコア logitsを取得 (vocab_size)
    probs = F.softmax(mask_logits, dim=-1)         # 確率化 (自分が解釈するため)
    top_probs, top_ids = torch.topk(probs, k=topk) # 上位 topk 個

    # topkの(ID，トークン，確率)を戻り値に指定
    candidates = []
    for prob, token_id in zip(top_probs.tolist(), top_ids.tolist()):
        token_str = tokenizer.id_to_token(token_id)
        candidates.append((token_id, token_str, prob))
    return {"position": pos,"candidates": candidates}


In [17]:
import torch.nn.functional as F

# トロイヤ戦争・黄金の
text = "神話ではトロイア戦争のきっかけは<mask>林檎を巡る"
# アルテミス
#text = "神話の中ではオレステースがイーピゲネイアと共にもたらした<mask>の神像は人身御供を要求する神であった"
# アポローン、鷹
#text = "神々は変身してエジプトへ逃げた時、アポローンは<mask>に、アレースは魚に、ヘーパイストスは雄牛に変身した。"
# ハーデス
#text = "アルテミスは後に神となるほどの腕前の医師アスクレーピオスを訪ね、オーリーオーンの復活を依頼したが、冥府の王<mask>がそれに異を唱えた。"
# 神
#text = "アルテミスは後に<mask>となるほどの腕前の医師アスクレーピオスを訪ね、オーリーオーンの復活を依頼した"#が、冥府の王<mask>がそれに異を唱えた。"



encoded = tokenizer.encode(text)
print(encoded.tokens)
results = predict_mask_topk(model, tokenizer, config, text, topk=5)

print(f"mask position = {results['position']}")
df = pd.DataFrame(results["candidates"] , columns=["id", "token", "prob"])
df

['神話', 'では', 'トロイア戦争', 'の', 'き', 'っ', 'かけ', 'は', '<mask>', '林', '檎', 'を', '巡', 'る']
mask position = 9


,id,token,prob
0,756,黄金の,0.984958
1,152,ゼウスの,0.001441
2,100,また,0.000913
3,78,ゼウスは,0.000818
4,961,青銅の,0.000805
